# Library Type Causal Analysis


In [ ]:
import warnings

import numpy as np
import pandas as pd
from dowhy import CausalModel
from scipy.stats import spearmanr
from sklearn.exceptions import DataConversionWarning
from sklearn.preprocessing import RobustScaler

warnings.filterwarnings("ignore", category=DataConversionWarning)

import sys
from pathlib import Path


def find_repo_root(start=Path.cwd()):
    for candidate in [start, *start.parents]:
        if (candidate / "utils" / "statistical_tests.py").exists():
            return candidate
    raise RuntimeError("Could not find repo root containing utils/statistical_tests.py")


REPO_ROOT = find_repo_root()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from utils import statistical_tests as tests


In [ ]:
INPUT_FILE = "alerts_with_lib_category_and_apk_size_semgrep.csv"


def normalize_verdict(value):
    normalized = str(value).strip().lower()
    if normalized in {"1", "true"}:
        return 1
    if normalized in {"0", "false"}:
        return 0
    raise ValueError(f"Unsupported verdict value: {value!r}")


df = pd.read_csv(INPUT_FILE)
print("Original rows:", len(df))

df = df[~df["code_location"].str.contains("obfuscated", case=False, na=False)].copy()
print("After removing obfuscated:", len(df))

df["verdict"] = df["verdict"].apply(normalize_verdict)

popularity_column = "app_popularity" if "app_popularity" in df.columns else "apk_category"
category_order = [
    "<100", "100-500", "500-1k", "1k-5k", "5k-10k", "10k-50k",
    "50k-100k", "100k-500k", "500k-1M", "1M-5M", ">5M",
]
ord_map = {category: index for index, category in enumerate(category_order)}
df["app_popularity_encoded"] = df[popularity_column].map(ord_map)
df = df.dropna(subset=["app_popularity_encoded", "apk_size"]).copy()
df["app_popularity_encoded"] = df["app_popularity_encoded"].astype(int)

scaler = RobustScaler()
df["apk_size_scaled"] = scaler.fit_transform(df[["apk_size"]])

print("Popularity column:", popularity_column)


In [ ]:
small_libs = ["SocialMedia", "Analytics", "Cloud"]
df["lib_grouped"] = df["code_location"].replace(small_libs, "Small_Libraries")
df.loc[df["lib_grouped"] == "developer_written", "lib_grouped"] = "App_Source_Code"

print("\nGrouped library counts:")
print(df["lib_grouped"].value_counts())

baseline_name = "App_Source_Code"
libs = [baseline_name] + [lib for lib in df["lib_grouped"].unique() if lib != baseline_name]
print("\nLibrary levels:")
for lib in libs:
    print("-", lib)


In [ ]:
def make_pairwise_balanced(df_in, lib_name, baseline="App_Source_Code", random_state=100):
    df_sub = df_in[df_in["lib_grouped"].isin([baseline, lib_name])].copy()
    baseline_df = df_sub[df_sub["lib_grouped"] == baseline]
    lib_df = df_sub[df_sub["lib_grouped"] == lib_name]
    target_count = len(baseline_df)

    if target_count == 0:
        raise ValueError("No App_Source_Code rows found for the baseline group.")
    if len(lib_df) == 0:
        raise ValueError(f"No rows found for library group: {lib_name}")

    if len(lib_df) >= target_count:
        lib_balanced = lib_df.sample(n=target_count, replace=False, random_state=random_state)
    else:
        lib_balanced = lib_df.sample(n=target_count, replace=True, random_state=random_state)

    df_balanced = pd.concat([baseline_df.copy(), lib_balanced], axis=0)
    return df_balanced.sample(frac=1, random_state=random_state).reset_index(drop=True)


def apply_refutation_tests(lib_name, label, model, identified_estimand, estimate, ate):
    ref_rows = []
    for method in refutation_methods:
        if method == "placebo_treatment_refuter":
            ref = model.refute_estimate(
                identified_estimand,
                estimate,
                method_name=method,
                placebo_type="permute",
                num_simulations=200,
            )
        elif method == "data_subset_refuter":
            ref = model.refute_estimate(
                identified_estimand,
                estimate,
                method_name=method,
                subset_fraction=0.8,
                num_simulations=200,
            )
        else:
            ref = model.refute_estimate(identified_estimand, estimate, method_name=method)

        print(f"\nRefuter: {method}\n", ref)
        ref_rows.append({
            "library": lib_name,
            "setting": label,
            "refuter": method,
            "orig_effect": ate,
            "new_effect": getattr(ref, "new_effect", None),
            "p_value": getattr(ref, "p_value", None),
        })
    return ref_rows


In [ ]:
def run_library_contrast(df_sub, lib_name, baseline="App_Source_Code", label="balanced"):
    df_sub = df_sub.copy()
    df_sub["treatment_lib"] = (df_sub["lib_grouped"] == lib_name).astype(int)

    print("\n" + "-" * 14 + f" {lib_name} vs {baseline} " + "-" * 14)

    # Combined approach: treated rows are baseline + selected library; control rows are baseline only.
    df_all = df_sub.copy()
    df_all["combined"] = 1
    df_baseline = df_sub[df_sub["treatment_lib"] == 0].copy()
    df_baseline["combined"] = 0
    df_sub = pd.concat([df_all, df_baseline], ignore_index=True)

    tests.correlation_test_treatment_to_outcome(df_sub)
    print(df_sub["combined"].value_counts())
    n0 = int((df_sub["combined"] == 0).sum())
    n1 = int((df_sub["combined"] == 1).sum())

    causal_graph = """
    digraph {
        combined -> verdict;
        app_popularity_encoded -> combined;
        app_popularity_encoded -> verdict;
        apk_size_scaled -> combined;
        apk_size_scaled -> verdict;
        app_popularity_encoded -> apk_size_scaled;
    }
    """

    model = CausalModel(
        data=df_sub,
        treatment="combined",
        outcome="verdict",
        graph=causal_graph.replace("\n", " "),
        common_causes=["app_popularity_encoded", "apk_size_scaled"],
    )

    identified_estimand = model.identify_effect(proceed_when_unidentifiable=True)
    estimate = model.estimate_effect(
        identified_estimand,
        method_name="backdoor.propensity_score_matching",
        target_units="ate",
        confidence_intervals="bootstrap",
        method_params={
            "num_simulations": 300,
            "sample_size_fraction": 1.0,
            "confidence_level": 0.95,
        },
    )

    ate = float(estimate.value)
    try:
        ci_low, ci_high = estimate.get_confidence_intervals()
        ci_low, ci_high = float(ci_low), float(ci_high)
    except Exception:
        ci_low, ci_high = None, None

    print("ATE:", ate)
    if ci_low is not None:
        print("95% CI:", (ci_low, ci_high))

    ref_rows = apply_refutation_tests(lib_name, label, model, identified_estimand, estimate, ate)

    result_row = {
        "library": lib_name,
        "setting": label,
        "n_baseline": n0,
        "n_treatment": n1,
        "ATE": ate,
        "CI_low": ci_low,
        "CI_high": ci_high,
    }
    return result_row, ref_rows


In [ ]:
main_rows = []
ref_rows_all = []

for lib in libs:
    if lib == baseline_name:
        continue

    df_balanced = make_pairwise_balanced(df, lib, baseline=baseline_name, random_state=100)
    main_row, ref_rows = run_library_contrast(df_balanced, lib, baseline=baseline_name, label="balanced")
    main_rows.append(main_row)
    ref_rows_all.extend(ref_rows)


In [ ]:
main_df = pd.DataFrame(main_rows).sort_values(["library", "setting"])
ref_df = pd.DataFrame(ref_rows_all)

print("\n\n", "-" * 25 + " Detailed (ATEs) " + "-" * 25)
print(main_df[["library", "setting", "n_baseline", "n_treatment", "ATE", "CI_low", "CI_high"]])

pivot = main_df.pivot(index="library", columns="setting", values="ATE")
print("\n\n", "-" * 25 + " Summary ATE " + "-" * 25)
print(pivot)
